# SMS spam classification with three models
DS483 · Student 2 · Tasks 3 and 4



In [1]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

SEED = 483
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
source = Path("splits.csv")
expected = "1d59db7174d98fce90fe16541e672c207fde0e05885269cbdb47f638b2ae0712"
assert hashlib.sha256(source.read_bytes().replace(b"\r\n", b"\n")).hexdigest() == expected, "Use Layan's shared splits.csv."
data = pd.read_csv(source, keep_default_na=False)
assert data.message_id.is_unique and data.groupby("clean_text").split.nunique().max() == 1
parts = {name: data.loc[data.split.eq(name)].reset_index(drop=True) for name in ("train", "validation", "test")}
y = {name: frame.label.eq("spam").astype(int).to_numpy() for name, frame in parts.items()}
print(pd.crosstab(data.split, data.label).to_string())
print("pandas", pd.__version__, "scikit-learn", sklearn.__version__, "torch", torch.__version__)

label        ham  spam
split                 
test         690   107
train       3447   533
validation   690   107
pandas 2.2.3 scikit-learn 1.8.0 torch 2.8.0+cpu


## Task 3 · Naive Bayes and Logistic Regression

Use the same sparse training TF-IDF matrix for both classifiers. Include single words and adjacent word pairs, with sublinear term frequency. Choose the parameter and spam cutoff with the highest validation spam F1. In ties keep the earlier candidate, with 0.50 first. A smaller Naive Bayes alpha means weaker smoothing. A larger logistic C means weaker regularization.

In [2]:
def choose_cutoff(labels, probabilities):
    candidates = [(f1_score(labels, probabilities >= t, zero_division=0), t)
                  for t in (0.50, 0.35, 0.65)]
    return max(candidates, key=lambda item: item[0])

tfidf = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                        ngram_range=(1, 2), sublinear_tf=True)
x_train = tfidf.fit_transform(parts["train"].clean_text)
x_val = tfidf.transform(parts["validation"].clean_text)
families = {
    "Naive Bayes": [(f"alpha={a}", MultinomialNB(alpha=a)) for a in (0.1, 0.5, 1.0)],
    "Logistic Regression": [(f"C={c}", LogisticRegression(C=c, solver="liblinear", random_state=SEED))
                            for c in (1.0, 4.0, 16.0)]
}
selected, val_rows = {}, []
for name, candidates in families.items():
    for setting, model in candidates:
        model.fit(x_train, y["train"])
        score, cutoff = choose_cutoff(y["validation"], model.predict_proba(x_val)[:, 1])
        val_rows.append((name, setting, cutoff, score))
        if name not in selected or score > selected[name]["f1"]:
            selected[name] = dict(model=model, setting=setting, cutoff=cutoff, f1=score)
print(pd.DataFrame(val_rows, columns=["model", "setting", "cutoff", "validation_spam_F1"]).round(4).to_string(index=False))

              model   setting  cutoff  validation_spam_F1
        Naive Bayes alpha=0.1    0.35              0.9665
        Naive Bayes alpha=0.5    0.35              0.8808
        Naive Bayes alpha=1.0    0.35              0.8387
Logistic Regression     C=1.0    0.35              0.9665
Logistic Regression     C=4.0    0.35              0.9668
Logistic Regression    C=16.0    0.35              0.9717


## Task 4 · Word sequences and LSTM

Use the same cleaned words, in their original order. Build a training-only vocabulary of words seen at least twice. Reserve IDs 0 and 1 for padding and unknown tokens. Use the training 99th percentile of lengths, with a ceiling of 100 words, and keep the first words of longer messages. Empty cleaned messages get one unknown token. The model learns a 32-dimensional embedding and 48 recurrent units. Max pooling over real time steps lets a cue near the beginning influence the prediction.

In [3]:
word_counts = Counter(word for text in parts["train"].clean_text for word in text.split())
vocab = {word: i for i, word in enumerate(sorted(w for w, n in word_counts.items() if n >= 2), start=2)}
lengths = parts["train"].clean_text.str.split().str.len()
max_len = max(1, min(100, int(np.ceil(np.percentile(lengths, 99)))))

def encode(messages):
    inputs = torch.zeros((len(messages), max_len), dtype=torch.long)
    for i, text in enumerate(messages):
        ids = [vocab.get(w, 1) for w in text.split()[:max_len]] or [1]
        inputs[i, :len(ids)] = torch.tensor(ids)
    return inputs

train_ids = encode(parts["train"].clean_text)
val_ids = encode(parts["validation"].clean_text)
print("Vocabulary (including special IDs):", len(vocab) + 2, "| maximum tokens:", max_len)

Vocabulary (including special IDs): 3192 | maximum tokens: 54


In [4]:
class SMSLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32, padding_idx=0)
        self.lstm = nn.LSTM(32, 48, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.output = nn.Linear(48, 1)

    def forward(self, ids):
        hidden, _ = self.lstm(self.embedding(ids))
        hidden = hidden.masked_fill(ids.eq(0).unsqueeze(-1), float("-inf"))
        pooled = hidden.max(dim=1).values
        return self.output(self.dropout(pooled)).squeeze(1)

def predict_probabilities(model, ids):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x).sigmoid() for x in ids.split(128)]).numpy()

In [5]:
network = SMSLSTM(len(vocab) + 2)
train_labels = torch.tensor(y["train"], dtype=torch.float32)
loader = DataLoader(TensorDataset(train_ids, train_labels), batch_size=64, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
positive_weight = ((train_labels == 0).sum() / train_labels.sum()).sqrt()
loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(network.parameters(), lr=0.003)
best_f1, best_epoch, history = -1.0, 0, []
for epoch in range(1, 21):
    network.train()
    total_loss = 0.0
    for ids, targets in loader:
        optimizer.zero_grad()
        loss = loss_fn(network(ids), targets)
        loss.backward()
        nn.utils.clip_grad_norm_(network.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(targets)
    val_f1, threshold = choose_cutoff(y["validation"], predict_probabilities(network, val_ids))
    history.append((epoch, total_loss / len(train_labels), val_f1, threshold))
    if val_f1 > best_f1:
        best_f1, best_epoch, best_cutoff = val_f1, epoch, threshold
        best_state = deepcopy(network.state_dict())
    if epoch - best_epoch >= 4:
        break
network.load_state_dict(best_state)
print(pd.DataFrame(history, columns=["epoch", "train_loss", "validation_spam_F1", "cutoff"]).round(4).to_string(index=False))
print("Selected epoch:", best_epoch, "| cutoff:", best_cutoff, "| trainable parameters:", sum(p.numel() for p in network.parameters()))

 epoch  train_loss  validation_spam_F1  cutoff
     1      0.5055              0.8485    0.35
     2      0.1972              0.8634    0.65
     3      0.1244              0.9273    0.35
     4      0.0853              0.9099    0.50
     5      0.0603              0.9340    0.50
     6      0.0490              0.9067    0.50
     7      0.0291              0.9050    0.35
     8      0.0199              0.9224    0.50
     9      0.0170              0.9124    0.50
Selected epoch: 5 | cutoff: 0.5 | trainable parameters: 117937


### Does token order affect the fitted LSTM?

For a validation-only diagnostic, shuffle the real tokens within each message using a fixed seed and keep the model, message lengths, padding and cutoff fixed. This measures sensitivity of this fitted model to that intervention. It does not isolate the causal benefit of word order over all competing training strategies.

In [6]:
shuffled = val_ids.clone()
rng = np.random.default_rng(SEED)
for row in shuffled:
    n = int(row.ne(0).sum())
    row[:n] = row[:n][rng.permutation(n)].clone()
ordered_pred = predict_probabilities(network, val_ids) >= best_cutoff
shuffled_pred = predict_probabilities(network, shuffled) >= best_cutoff
print("Validation spam F1 (original, shuffled):",
      round(f1_score(y["validation"], ordered_pred), 4), round(f1_score(y["validation"], shuffled_pred), 4))
print("Changed validation predictions:", int((ordered_pred != shuffled_pred).sum()))

Validation spam F1 (original, shuffled): 0.934 0.9163
Changed validation predictions: 9


## Shared test evaluation

Freeze all choices before scoring the 797 test messages. Spam is positive. FP counts legitimate messages flagged as spam, while FN counts spam delivered as legitimate. Export exact scores and predictions indexed by the original message ID.

In [7]:
test = parts["test"]
x_test = tfidf.transform(test.clean_text)
estimates = {name: (entry["model"].predict_proba(x_test)[:, 1], entry["cutoff"], entry["setting"])
             for name, entry in selected.items()}
estimates["LSTM"] = (predict_probabilities(network, encode(test.clean_text)), best_cutoff, f"epoch={best_epoch}")
results, prediction_rows = [], []
for name, (prob, cutoff, setting) in estimates.items():
    pred = (prob >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(y["test"], pred, labels=[0, 1]).ravel()
    results.append(dict(model=name, setting=setting, cutoff=cutoff, test_size=len(test),
                        accuracy=accuracy_score(y["test"], pred), spam_precision=precision_score(y["test"], pred),
                        spam_recall=recall_score(y["test"], pred), spam_f1=f1_score(y["test"], pred),
                        macro_f1=f1_score(y["test"], pred, average="macro"), TN=tn, FP=fp, FN=fn, TP=tp))
    prediction_rows.append(pd.DataFrame(dict(message_id=test.message_id, actual=test.label,
        model=name, predicted=np.where(pred, "spam", "ham"), spam_score=prob)))
metrics = pd.DataFrame(results)
predictions = pd.concat(prediction_rows, ignore_index=True)
metrics.to_csv("student2_metrics.csv", index=False)
predictions.to_csv("student2_predictions.csv", index=False)
print(metrics.round(4).to_string(index=False))

              model   setting  cutoff  test_size  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
        Naive Bayes alpha=0.1    0.35        797    0.9875          0.9619       0.9439   0.9528    0.9728 686   4   6 101
Logistic Regression    C=16.0    0.35        797    0.9812          0.9107       0.9533   0.9315    0.9603 680  10   5 102
               LSTM   epoch=5    0.50        797    0.9787          0.8879       0.9626   0.9238    0.9557 677  13   4 103


## Inspect errors by model

For each classifier, show the first false alarm and first missed spam by source-line order, if present. These are examples, not estimates of error frequency. Inspect the counts above for frequency.

In [8]:
mistakes = predictions[predictions.actual != predictions.predicted].merge(
    test[["message_id", "text"]], on="message_id", validate="many_to_one")
examples = mistakes.sort_values("message_id").groupby(["model", "actual"], sort=True).head(1)
for row in examples.sort_values(["model","actual"]).itertuples():
    print(f"{row.model} | {row.message_id} | {row.actual} → {row.predicted} | score {row.spam_score:.3f}")
    print(row.text, "\n")

LSTM | sms_00426 | ham → spam | score 0.716
Send this to ur friends and receive something about ur voice..... How is my speaking expression? 1.childish 2.naughty 3.Sentiment 4.rowdy 5.ful of attitude 6.romantic 7.shy 8.Attractive 9.funny  &lt;#&gt; .irritating  &lt;#&gt; .lovable. reply me.. 

LSTM | sms_00166 | spam → ham | score 0.451
BangBabes Ur order is on the way. U SHOULD receive a Service Msg 2 download UR content. If U do not, GoTo wap. bangb. tv on UR mobile internet/service menu 

Logistic Regression | sms_00426 | ham → spam | score 0.436
Send this to ur friends and receive something about ur voice..... How is my speaking expression? 1.childish 2.naughty 3.Sentiment 4.rowdy 5.ful of attitude 6.romantic 7.shy 8.Attractive 9.funny  &lt;#&gt; .irritating  &lt;#&gt; .lovable. reply me.. 

Logistic Regression | sms_02559 | spam → ham | score 0.082
This message is brought to you by GMW Ltd. and is not connected to the 

Naive Bayes | sms_01291 | ham → spam | score 0.652
Hey...Grea

## Interpreting the trade-off

Naive Bayes had the highest test spam F1 (0.9528). Logistic Regression caught one more spam message but incorrectly flagged six more legitimate ones. The LSTM caught another spam message beyond Logistic Regression and added three false alarms. The token-order shuffle changed nine validation predictions and reduced spam F1 from 0.9340 to 0.9163. Word order mattered to this fitted network; its test spam F1 still trailed both classical baselines.

## Reading the error examples

The farm-tour invitation (sms_01291) is legitimate but includes prices and a deposit, patterns also present in commercial messages. The two classical models missed the truncated company disclaimer (sms_02559). The chain-style message sms_00426 is labelled ham and was flagged by both Logistic Regression and the LSTM. The LSTM missed the order-style message sms_00166 with a spam score just below its cutoff. Textual explanations are hypotheses, not feature attributions.

## Handoff for the BERT comparison

Use the same split file and treat spam as positive. Keep training, checkpoint selection and threshold selection separate from test scoring. Join BERT predictions to student2_predictions.csv by message_id. Compare precision, recall, spam F1, FP and FN.